In [1]:
# ─────────────────────────────────────────────────────────────────────
# NOTEBOOK 05a · CELL 1 · SETUP AND THE LEAGUE TABLE (does our points logic reproduce real playoff lines?)
# Purpose: some matches matter more than others. A league match in week one barely changes a team's season; the last league
#          match for the 4th place, or a final, changes everything. We measure HOW MUCH each match matters (its "stakes"),
#          separately from how tense a moment inside the match is (that is the Leverage Index, notebook 05b).
# ─────────────────────────────────────────────────────────────────────
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80); pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
OUT_DIR = Path("interim"); SEED = 42

def load_table(name):
    return pd.read_parquet(OUT_DIR / f"{name}.parquet") if (OUT_DIR / f"{name}.parquet").exists() else pd.read_pickle(OUT_DIR / f"{name}.pkl")
def save_table(df, name):
    try:
        df.to_parquet(OUT_DIR / f"{name}.parquet", index=False); return "parquet"
    except Exception:
        df.to_pickle(OUT_DIR / f"{name}.pkl"); return "pickle"

matches = load_table("matches_clean")
ball    = load_table("ball_master")
teams   = load_table("teams")

# 1a) The IPL fixtures, in the order they were played. match_number is the season's own fixture number.
# FUNCTIONAL: IPL has a league stage (every team plays 14 matches, win = 2 points, no result = 1 point each, ties are settled
# by a super over so someone still wins) and then four playoff matches. We rebuild the league table from results alone.
ipl = matches[matches.tier == "IPL"].sort_values(["year", "match_number"]).reset_index(drop=True)
ipl["team_a"], ipl["team_b"] = ipl.home_team_id.astype(int), ipl.away_team_id.astype(int)
name = (pd.concat([matches[["home_team_id", "home_team"]].set_axis(["id", "n"], axis=1), matches[["away_team_id", "away_team"]].set_axis(["id", "n"], axis=1)])
          .dropna().drop_duplicates("id").set_index("id").n)
league = ipl[ipl.stage == "League"].copy()
print("League matches per season:", league.groupby("year").size().to_dict(), "| playoff matches per season:", ipl[ipl.stage != "League"].groupby("year").size().to_dict())
cnt = league.groupby("year").apply(lambda d: pd.concat([d.team_a, d.team_b]).value_counts().agg(["min", "max"]))
print("Matches played per team (min, max) by season:"); display(cnt.T)
print("2025 note: two teams (Delhi, Punjab) are recorded with 15 league matches, probably because an abandoned fixture was replayed. We keep the data as recorded.")

# 1b) Net run rate (NRR) from the ball data: the tie-break the league actually uses.
# TECHNICAL: NRR = runs scored per over faced minus runs conceded per over bowled. A side that is bowled out is charged its FULL
# quota of overs. We ignore the small DLS adjustments of rain-hit games.
inn = ball[ball.match_id.isin(league.match_id)].groupby(["match_id", "innings", "batting_team_id", "bowling_team_id"]).agg(
    runs=("total_runs", "sum"), legal=("is_legal", "sum"), wk=("is_wicket_clean", "sum"), quota=("innings_balls", "first")).reset_index()
inn["overs"] = np.where(inn.wk >= 10, inn.quota, inn.legal) / 6
nrr = (inn.merge(league[["match_id", "year"]], on="match_id")
          .pipe(lambda d: pd.concat([d.rename(columns={"batting_team_id": "team", "runs": "rf", "overs": "of"}).assign(ra=0.0, oa=0.0)[["year", "team", "rf", "of", "ra", "oa"]],
                                     d.rename(columns={"bowling_team_id": "team", "runs": "ra", "overs": "oa"}).assign(rf=0.0, of=0.0)[["year", "team", "rf", "of", "ra", "oa"]]]))
          .groupby(["year", "team"]).sum())
nrr["nrr"] = nrr.rf / nrr["of"] - nrr.ra / nrr.oa
nrr = nrr.nrr

# 1c) The final league table of every season: points, then NRR. Compare with the four teams that really reached the playoffs.
def final_table(d):
    pts = {}
    for r in d.itertuples():
        pts.setdefault(r.team_a, 0); pts.setdefault(r.team_b, 0)
        if pd.isna(r.win_team_id): pts[r.team_a] += 1; pts[r.team_b] += 1
        else: pts[int(r.win_team_id)] += 2
    return pd.Series(pts)
rows = []
for y, d in league.groupby("year"):
    t = final_table(d).to_frame("points"); t["nrr"] = nrr.loc[y].reindex(t.index)
    t = t.sort_values(["points", "nrr"], ascending=False); t["pos"] = range(1, len(t) + 1)
    po = ipl[(ipl.year == y) & (ipl.stage != "League")]
    playoff_teams = set(po.team_a) | set(po.team_b)
    q1 = ipl[(ipl.year == y) & (ipl.stage == "Qualifier")].sort_values("match_number").iloc[0]
    rows.append({"year": y, "top 4 by points then NRR": ", ".join(name[t.index[:4]]),
                 "same four as the real playoffs": set(t.index[:4]) == playoff_teams,
                 "Qualifier 1 teams are the top 2": {q1.team_a, q1.team_b} == set(t.index[:2]),
                 "4th/5th place points": f"{t.points.iloc[3]} / {t.points.iloc[4]}"})
chk = pd.DataFrame(rows).set_index("year"); display(chk)
assert chk["same four as the real playoffs"].all(), "league table logic does not reproduce the real playoff teams"
print("Check passed: in all five seasons the table rebuilt from results gives exactly the four real playoff teams.")

League matches per season: {2022: 70, 2023: 70, 2024: 70, 2025: 71, 2026: 70} | playoff matches per season: {2022: 4, 2023: 4, 2024: 4, 2025: 4, 2026: 4}
Matches played per team (min, max) by season:


year,2022,2023,2024,2025,2026
min,14,14,14,14,14
max,14,14,14,15,14


2025 note: two teams (Delhi, Punjab) are recorded with 15 league matches, probably because an abandoned fixture was replayed. We keep the data as recorded.


,top 4 by points then NRR,same four as the real playoffs,Qualifier 1 teams are the top 2,4th/5th place points
year,,,,
2022,"Gujarat Titans, Rajasthan Royals, Lucknow Supe...",True,True,16 / 14
2023,"Gujarat Titans, Chennai Super Kings, Lucknow S...",True,True,16 / 14
2024,"Kolkata Knight Riders, Sunrisers Hyderabad, Ra...",True,True,14 / 14
2025,"Punjab Kings, Royal Challengers Bengaluru, Guj...",True,True,16 / 16
2026,"Royal Challengers Bengaluru, Gujarat Titans, S...",True,True,16 / 15


Check passed: in all five seasons the table rebuilt from results gives exactly the four real playoff teams.


### Cell 1 · Setup and the league table check

**Procedure**
1. Listed every IPL fixture in playing order and rebuilt each season's league table from results alone (win = 2 points, no result = 1 each).
2. Computed net run rate (NRR) from the ball data, charging a bowled-out side its full overs, and used it as the tie-break.
3. Compared the top four of the rebuilt table with the four teams that really played the playoffs.

**Observations**
- Each season has 70 league matches (71 in 2025) and 4 playoff matches. Every team plays 14, except two teams in 2025 with 15, probably because an abandoned fixture was replayed. Kept as recorded.
- In all five seasons the rebuilt table gives exactly the four real playoff teams, and Qualifier 1 is always played by the top two (assertion passed).
- This includes 2024, where Bengaluru, Chennai and Delhi were tied on 14 points for the last place, and NRR correctly put Bengaluru through. The points and tie-break logic is sound, so the simulations that follow start from correct tables.

In [2]:
# ─────────────────────────────────────────────────────────────────────
# CELL 2 · THE SEASON SIMULATOR: how much does ONE league match change a team's season?
# ─────────────────────────────────────────────────────────────────────
# FUNCTIONAL (the idea). Before a league match, ask: "if this team WINS, what are its chances of finishing in the top 4 (and the
# top 2)? And if it LOSES?" The gap between the two answers is how much the match matters to that team. A team that is already
# through, or already out, has almost no gap; a team fighting for 4th place has a big one. We turn that gap into one currency,
# the change in the chance of WINNING THE TITLE, so league and playoff matches can be put on the same scale (Cell 3).
#
# TECHNICAL (how). 1) Take the real points table just before the match. 2) Play out the REST of the season 4,000 times, with
# every remaining match a coin flip. 3) Do this twice, once with the team winning this match and once losing it, using the SAME
# random remaining results both times (this "common random numbers" trick removes most of the simulation noise from the
# difference). 4) Count how often the team finishes top 4 and top 2. Ties on points are broken at random (the real tie-break, net
# run rate, depends on future margins we cannot know).
# Title chance from the league position, assuming four equal playoff teams: a top-2 side has 37.5% (it gets two lives), a 3rd or
# 4th side has 12.5% (it must win the eliminator), so  P(title) = 0.125 x P(top 4) + 0.25 x P(top 2).
N_SIMS = 4000
rng = np.random.default_rng(SEED)

def simulate_season(d, win_prob=None, n_sims=N_SIMS, rng=rng):
    """d = the league matches of ONE season in playing order. Returns one row per match x team with the stake."""
    ids = sorted(set(d.team_a) | set(d.team_b)); ix = {t: k for k, t in enumerate(ids)}; T = len(ids)
    A = d.team_a.map(ix).to_numpy(); B = d.team_b.map(ix).to_numpy(); n = len(d)
    won_a = np.where(d.win_team_id.isna(), -1, (d.win_team_id == d.team_a).astype(int)).tolist()   # -1 = no result
    pts = np.zeros(T)                                                    # real points table, updated match by match
    out = []
    for k in range(n):
        rem = np.arange(k + 1, n)
        # play the rest of the season: each remaining match goes to team A with probability p (0.5 unless we pass strengths)
        p = 0.5 if win_prob is None else win_prob[k + 1:]
        a_wins = rng.random((n_sims, len(rem))) < p
        winner = np.where(a_wins, A[rem], B[rem])
        rest = np.stack([2 * (winner == t).sum(1) for t in range(T)], axis=1)         # points won in the remaining matches
        tie_noise = rng.random((n_sims, T)) * 0.9                                     # random tie-break (never reorders unequal points)
        res = {}
        for who, label in [(A[k], "win"), (B[k], "win_b")]:                           # team `who` wins match k
            add = np.zeros(T); add[who] = 2
            tot = pts + add + rest + tie_noise
            rank = (-tot).argsort(1).argsort(1)
            res[(who, label)] = ((rank < 4).mean(0), (rank < 2).mean(0))
        # the stake of the match for each of the two teams: result when it wins minus result when it loses
        for team, other in [(A[k], B[k]), (B[k], A[k])]:
            w4, w2 = res[(team, "win" if team == A[k] else "win_b")]
            l4, l2 = res[(other, "win" if other == A[k] else "win_b")]               # the OTHER team winning = this team losing
            title = lambda p4, p2: 0.125 * p4 + 0.25 * p2
            out.append({"match_id": d.match_id.iloc[k], "team": ids[team], "p4_if_win": w4[team], "p4_if_loss": l4[team],
                        "p2_if_win": w2[team], "p2_if_loss": l2[team],
                        "title_if_win": title(w4[team], w2[team]), "title_if_loss": title(l4[team], l2[team])})
        # now apply the REAL result of match k to the points table before moving on
        if won_a[k] == -1: pts[A[k]] += 1; pts[B[k]] += 1
        else: pts[A[k] if won_a[k] == 1 else B[k]] += 2
    o = pd.DataFrame(out)
    o["p4_before"] = (o.p4_if_win + o.p4_if_loss) / 2          # chance of the top 4 before the match, if it were a coin flip
    o["stake"] = o.title_if_win - o.title_if_loss              # change in title chance between winning and losing (>= 0)
    o["stake_top4"] = o.p4_if_win - o.p4_if_loss               # change in top-4 chance, easier to read
    return o

# 2a) Run all five seasons. About 70 matches x 2 teams per season, 4,000 simulated remainders each.
import time; t0 = time.time()
sim = pd.concat([simulate_season(d.reset_index(drop=True)).assign(year=y) for y, d in league.groupby("year")], ignore_index=True)
print(f"Simulated {len(sim)//2} league matches in {time.time()-t0:.0f}s.")
assert (sim.stake > -0.03).all(), "winning should not lower the title chance beyond simulation noise"

# 2b) Sanity checks.
# (1) At the start of a season, every team has about a 40% chance of the top 4 (4 places for 10 equal teams).
first = sim.merge(league[["match_id", "match_number", "year"]], on=["match_id", "year"]).query("match_number <= 2")
print(f"Top-4 chance before the very first matches (should be near 0.40): {first.groupby('year').p4_before.mean().round(3).to_dict()}")
# (2) Winning helps, losing hurts: the top-4 chance if the team wins must be above the chance if it loses.
print(f"Share of match-team rows where winning gives a higher top-4 chance than losing: {(sim.p4_if_win >= sim.p4_if_loss - 0.02).mean():.1%}")
display(sim[["p4_before", "stake_top4", "stake"]].describe().round(3).T)

Simulated 351 league matches in 4s.
Top-4 chance before the very first matches (should be near 0.40): {2022: 0.403, 2023: 0.398, 2024: 0.399, 2025: 0.374, 2026: 0.396}
Share of match-team rows where winning gives a higher top-4 chance than losing: 100.0%


,count,mean,std,min,25%,50%,75%,max
p4_before,702.000,0.401,0.280,0.000,0.174,0.386,0.598,1.000
stake_top4,702.000,0.192,0.120,0.000,0.121,0.202,0.244,1.000
stake,702.000,0.060,0.038,0.000,0.035,0.062,0.081,0.250


### Cell 2 · Season simulator: how much does one league match change a season?

**Procedure**
1. Took the real points table just before each league match.
2. Played out the rest of the season 4,000 times with every remaining match a coin flip, once assuming the team wins this match and once assuming it loses, using the same random results for both.
3. Counted how often the team finishes in the top 4 and the top 2, with random tie-breaks, and converted each into the chance of winning the title: P(title) = 0.125 x P(top 4) + 0.25 x P(top 2).
4. The stake of a match for a team is the difference between its title chance if it wins and if it loses.

**Observations**
- Before the first matches, each team's chance of the top 4 is about 0.40 in every season (4 places for 10 teams). 2025 is 0.374 because of its extra fixtures.
- Winning raised the top-4 chance in 100% of cases, so the logic points the right way.
- A typical match moves a team's top-4 chance by about 0.19 (up to a full 1.0 in a last-chance game) and its title chance by 0.06 (maximum 0.25 for a league match).

In [3]:
# ─────────────────────────────────────────────────────────────────────
# CELL 3 · THE PLAYOFF LADDER AND THE FULL STAKES TABLE (every match, both teams)
# ─────────────────────────────────────────────────────────────────────
# FUNCTIONAL: the same currency (change in title chance, "title points") for playoffs, using the IPL format you described:
#   Qualifier 1  (P1 v P2): winner -> Final (title 50%), loser -> Qualifier 2 (title 25%)           stake 0.25
#   Eliminator   (P3 v P4): winner -> Qualifier 2 (title 25%), loser is OUT (0%)                      stake 0.25
#   Qualifier 2  (loser Q1 v winner Elim): winner -> Final (50%), loser OUT (0%)                      stake 0.50
#   Final: winner 100%, loser 0%                                                                      stake 1.00
# (every playoff team is assumed equally strong, so each coin-flip match halves the chances).
# Qualifier 1 is NOT sudden death: the loser gets a second life, so it is worth less than the Eliminator in "elimination" terms,
# but equal in title terms. Using the title-chance scale captures this without any hand-picked weights.
LADDER = {"Qualifier 1": 0.25, "Eliminator": 0.25, "Qualifier 2": 0.50, "Final": 1.00}
ipl["ladder"] = ipl.stage
q = ipl[ipl.stage == "Qualifier"].groupby("year").match_number.rank(method="first")      # 1st Qualifier of a season = Q1, 2nd = Q2
ipl.loc[q.index, "ladder"] = np.where(q == 1, "Qualifier 1", "Qualifier 2")
ipl_po = ipl[ipl.stage != "League"].copy(); ipl_po["stake"] = ipl_po.ladder.map(LADDER)
display(ipl_po.groupby(["ladder"]).agg(matches=("match_id", "size"), stake=("stake", "first")).loc[list(LADDER)])

# 3a) One row per match x team for EVERY match in the data.
# League matches: the simulated stake. IPL playoffs: the ladder. T20I and A-team tiers have no league table to simulate, so only
# their sudden-death games are measured (a semi-final is worth 0.50, a final 1.00); other matches there are "not measured" and
# will be treated as neutral (multiplier 1) in the pressure model.
allm = matches[["match_id", "tier", "year", "stage", "match_number", "home_team_id", "away_team_id", "win_team_id"]].copy()
rows = pd.concat([allm.assign(team=allm.home_team_id), allm.assign(team=allm.away_team_id)], ignore_index=True)
rows = rows.merge(sim[["match_id", "team", "p4_before", "stake", "stake_top4"]], on=["match_id", "team"], how="left")
rows["kind"] = np.select([rows.match_id.isin(sim.match_id), rows.match_id.isin(ipl_po.match_id)], ["league", "playoff"], default="other")
ladder_stake = rows.match_id.map(ipl_po.set_index("match_id").stake)
other_ko = rows.tier.ne("IPL") & rows.stage.map({"Final": 1.0, "Semi Final": 0.5}).notna()
rows["stake"] = np.where(rows.kind == "playoff", ladder_stake, rows.stake)
ko_stake = rows.stage.map({"Final": 1.0, "Semi Final": 0.5})
rows.loc[other_ko, "stake"] = ko_stake[other_ko]
rows.loc[other_ko, "kind"] = "knockout"
rows["stake_known"] = rows.stake.notna()
rows["ladder"] = rows.match_id.map(ipl.set_index("match_id").ladder)

# 3b) For league matches: which match of the team's 14 is it, and how many does it still have after this one?
lg = rows[rows.kind == "league"].sort_values(["year", "team", "match_number"])
lg["team_match_no"] = lg.groupby(["year", "team"]).cumcount() + 1
lg["matches_left"] = lg.groupby(["year", "team"]).team_match_no.transform("max") - lg.team_match_no
rows = rows.merge(lg[["match_id", "team", "team_match_no", "matches_left"]], on=["match_id", "team"], how="left")
print(f"\nStakes table: {len(rows):,} match-team rows. Measured: {rows.stake_known.sum():,} ({rows.stake_known.mean():.0%}); by kind:")
display(rows.groupby("kind").agg(rows=("match_id", "size"), measured=("stake_known", "sum"), mean_stake=("stake", "mean"), max_stake=("stake", "max")).round(3))

,matches,stake
ladder,,
Qualifier 1,5,0.250
Eliminator,5,0.250
Qualifier 2,5,0.500
Final,5,1.000



Stakes table: 1,766 match-team rows. Measured: 764 (43%); by kind:


,rows,measured,mean_stake,max_stake
kind,,,,
knockout,22,22,0.818,1.000
league,702,702,0.060,0.250
other,1002,0,NaN,NaN
playoff,40,40,0.500,1.000


### Cell 3 · Playoff ladder and the full stakes table

**Procedure**
1. Valued the four IPL playoff matches in the same title-chance currency: Qualifier 1 = 0.25, Eliminator = 0.25, Qualifier 2 = 0.50, Final = 1.00 (equal-strength teams assumed).
2. For the T20I and A-team tiers, which have no league table, measured only their sudden-death games: semi-final = 0.50, final = 1.00.
3. Built one row per match and team for every match, with the stake, the kind of match, which fixture of the team's 14 it is, and how many it has left.

**Observations**
- 764 of 1,766 match-team rows are measured: 702 league, 40 IPL playoff and 22 T20I or A-team semi-finals and finals. Everything else (1,002 rows) is "not measured" and will be treated as neutral in the pressure model.
- Qualifier 1 is worth the same as the Eliminator in title terms, even though only the Eliminator loser is knocked out. A team that loses Qualifier 1 keeps a second life, so its title chance only halves.
- Playoff and final matches have a mean stake of 0.50 and 0.82. Every league match is below 0.25, so a final matters at least four times as much as any league game.

In [4]:
# ─────────────────────────────────────────────────────────────────────
# CELL 4 · TESTING THE "RUN-IN" IDEA: are the last 4 to 6 league matches really the ones that matter?
# ─────────────────────────────────────────────────────────────────────
# FUNCTIONAL: the hypothesis was that the last 4-6 matches before the playoffs are near do-or-die. Our stake numbers let us TEST it
# instead of assuming it. We group each team's 14 league matches by how many it still has to play AFTER the one in question.
league_rows = rows[rows.kind == "league"].copy()
league_rows["block"] = pd.cut(league_rows.matches_left, [-1, 3, 5, 8, 20], labels=["last 4 matches", "5th and 6th from last", "middle", "opening"])
league_rows["live"] = league_rows.p4_before.between(0.05, 0.95)          # a team that is neither almost certainly in nor almost certainly out
by = league_rows.groupby("block").agg(team_matches=("stake", "size"), mean_title_stake=("stake", "mean"), mean_top4_swing=("stake_top4", "mean"),
                                      share_live=("live", "mean"), high_stake_share=("stake", lambda s: (s >= 0.10).mean()))
display(by.loc[["opening", "middle", "5th and 6th from last", "last 4 matches"]].round(3))

# By number of matches left, one column each (14 matches played by every team, so 13 left = the team's first match).
display(league_rows.groupby("matches_left").agg(team_matches=("stake", "size"), mean_title_stake=("stake", "mean"), share_live=("live", "mean"),
                                                 share_over_0_10=("stake", lambda s: (s >= 0.10).mean())).round(3).T)

# 4a) Does the pattern hold every season? (Some seasons are decided early, others go to the last day.)
display(league_rows.groupby(["year", "block"]).stake.mean().unstack().loc[:, ["opening", "middle", "5th and 6th from last", "last 4 matches"]].round(3))

# 4b) The most decisive league matches in five seasons (both teams' stakes averaged), as a common-sense check.
mm = (league_rows.groupby("match_id").agg(year=("year", "first"), match_number=("match_number", "first"), stake=("stake", "mean"), top4_swing=("stake_top4", "mean"))
        .join(matches.set_index("match_id")[["home_team", "away_team", "match_date"]]).sort_values("stake", ascending=False))
print("\nTen most decisive league matches (average title-chance swing for the two teams):")
display(mm.head(10)[["year", "match_number", "match_date", "home_team", "away_team", "stake", "top4_swing"]].round(3))
print("\nThe LEAST decisive (typically early season, or a team already through / out):")
display(mm.tail(5)[["year", "match_number", "match_date", "home_team", "away_team", "stake", "top4_swing"]].round(3))

,team_matches,mean_title_stake,mean_top4_swing,share_live,high_stake_share
block,,,,,
opening,252,0.061,0.200,1.000,0.000
middle,150,0.061,0.196,0.927,0.027
5th and 6th from last,100,0.059,0.191,0.770,0.180
last 4 matches,200,0.060,0.181,0.505,0.280


matches_left,0.000,1.000,2.000,3.000,4.000,5.000,6.000,7.000,8.000,9.000,10.000,11.000,12.000,13.000,14.000
team_matches,50.000,50.000,50.000,50.000,50.000,50.000,50.000,50.000,50.000,50.000,50.000,50.000,50.000,50.000,2.000
mean_title_stake,0.057,0.061,0.058,0.065,0.060,0.058,0.059,0.062,0.061,0.062,0.060,0.060,0.060,0.060,0.066
share_live,0.300,0.420,0.540,0.760,0.780,0.760,0.920,0.900,0.960,1.000,1.000,1.000,1.000,1.000,1.000
share_over_0_10,0.280,0.280,0.260,0.300,0.260,0.100,0.080,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000


block,opening,middle,5th and 6th from last,last 4 matches
year,,,,
2022,0.060,0.063,0.053,0.050
2023,0.062,0.067,0.078,0.069
2024,0.061,0.059,0.053,0.050
2025,0.060,0.058,0.056,0.061
2026,0.059,0.056,0.055,0.071



Ten most decisive league matches (average title-chance swing for the two teams):


,year,match_number,match_date,home_team,away_team,stake,top4_swing
match_id,,,,,,,
20243,2023,63,2023-05-16,Lucknow Super Giants,Mumbai Indians,0.187,0.500
40754,2025,70,2025-05-26,Punjab Kings,Mumbai Indians,0.156,0.246
50377,2026,56,2026-05-12,Gujarat Titans,Sunrisers Hyderabad,0.154,0.280
14157,2022,63,2022-05-15,Lucknow Super Giants,Rajasthan Royals,0.150,0.203
50384,2026,63,2026-05-18,Chennai Super Kings,Sunrisers Hyderabad,0.139,0.473
50373,2026,52,2026-05-09,Rajasthan Royals,Gujarat Titans,0.136,0.380
40749,2025,56,2025-05-06,Mumbai Indians,Gujarat Titans,0.128,0.361
50388,2026,67,2026-05-22,Sunrisers Hyderabad,Royal Challengers Bengaluru,0.125,0.000
40752,2025,71,2025-05-27,Lucknow Super Giants,Royal Challengers Bengaluru,0.125,0.000



The LEAST decisive (typically early season, or a team already through / out):


,year,match_number,match_date,home_team,away_team,stake,top4_swing
match_id,,,,,,,
14150,2022,56,2022-05-09,Mumbai Indians,Kolkata Knight Riders,0.001,0.010
40756,2025,63,2025-05-20,Chennai Super Kings,Rajasthan Royals,0.000,0.000
14164,2022,70,2022-05-22,Sunrisers Hyderabad,Punjab Kings,0.000,0.000
40753,2025,69,2025-05-25,Sunrisers Hyderabad,Kolkata Knight Riders,0.000,0.000
50391,2026,70,2026-05-24,Kolkata Knight Riders,Delhi Capitals,0.000,0.000


### Cell 4 · Testing the "last 4 to 6 matches" idea

**Procedure**
1. Grouped each team's league matches by how many it still has to play after that match: opening (9+ left), middle (6 to 8), the 5th and 6th from last, and the last 4.
2. For each group: average stake, average top-4 swing, share of teams still "live" (top-4 chance between 5% and 95%), and the share of matches with a stake of 0.10 or more.
3. Repeated by season and listed the most and least decisive league matches.

**Observations**
- The average stake is about 0.06 in every block, so the last matches are not more important on average. What changes is the shape: 0% of opening matches have a stake of 0.10 or more, against 18% for the 5th and 6th from last and 28% for the last 4.
- Late in the season, matches split in two. Roughly 28% are decisive, but the share of teams still live falls from 100% to 50%, so many late games are dead rubbers for one or both sides.
- It varies by season: 2023, 2025 and 2026 have late-season stakes above the early ones (2023: 0.069 to 0.078, 2026: 0.071), while 2022 and 2024 were settled earlier.
- The most decisive matches are the famous ones: Lucknow v Mumbai (16 May 2023, a pure knockout for 4th place, top-4 swing of 0.50), Punjab v Mumbai (26 May 2025), Chennai v Hyderabad (18 May 2026), Rajasthan v Gujarat (9 May 2026).
- Matches with a top-4 swing of 0 but a stake of 0.125 (Hyderabad v Bengaluru 2026, Lucknow v Bengaluru 2025, Rajasthan v Kolkata 2024) are top-2 fights between teams already through. This is why the title-chance scale is better than a "do or die" flag.
- The least decisive are late-season matches between teams already through or already out.

In [5]:
# ─────────────────────────────────────────────────────────────────────
# CELL 5 · ARE THE STAKES TRUSTWORTHY? Two robustness checks
# ─────────────────────────────────────────────────────────────────────
# 5a) SIMULATION NOISE. Re-run the whole thing with another random seed. If the stakes were mostly noise, the two runs would
# disagree. The agreement tells us how much of the number is real.
sim2 = pd.concat([simulate_season(d.reset_index(drop=True), rng=np.random.default_rng(7)).assign(year=y) for y, d in league.groupby("year")], ignore_index=True)
cmp = sim[["match_id", "team", "stake"]].merge(sim2[["match_id", "team", "stake"]], on=["match_id", "team"], suffixes=("_a", "_b"))
print(f"Two simulation runs: correlation of match stakes = {cmp.stake_a.corr(cmp.stake_b):.3f}; average absolute difference = {(cmp.stake_a - cmp.stake_b).abs().mean():.4f} title points "
      f"(typical stake {cmp.stake_a.mean():.3f}).")

# 5b) TEAM STRENGTH. The simulator treats every remaining match as a coin flip. A fairer test uses team strength: here, the Elo
# rating from notebook 02 (built from earlier matches only). If stakes barely change, the coin-flip simplification is harmless.
first_ball = ball[ball.innings == 1].groupby("match_id").agg(elo_bat=("bat_elo", "first"), elo_bowl=("bowl_elo", "first"))
bat_team = ball[ball.innings == 1].groupby("match_id").batting_team_id.first(); bowl_team = ball[ball.innings == 1].groupby("match_id").bowling_team_id.first()
elo = pd.concat([pd.DataFrame({"match_id": first_ball.index, "team": bat_team.reindex(first_ball.index).to_numpy(), "elo": first_ball.elo_bat.to_numpy()}),
                 pd.DataFrame({"match_id": first_ball.index, "team": bowl_team.reindex(first_ball.index).to_numpy(), "elo": first_ball.elo_bowl.to_numpy()})])
elo_map = elo.set_index(["match_id", "team"]).elo
def elo_win_prob(d):
    ea = np.array([elo_map.get((m, t), 1500.0) for m, t in zip(d.match_id, d.team_a)])
    eb = np.array([elo_map.get((m, t), 1500.0) for m, t in zip(d.match_id, d.team_b)])
    return 1 / (1 + 10 ** ((eb - ea) / 400))          # standard Elo formula: 400 points of rating gap = 10 to 1 odds
sim3 = pd.concat([simulate_season(d.reset_index(drop=True), win_prob=elo_win_prob(d.reset_index(drop=True))).assign(year=y) for y, d in league.groupby("year")], ignore_index=True)
cmp3 = sim[["match_id", "team", "stake"]].merge(sim3[["match_id", "team", "stake"]], on=["match_id", "team"], suffixes=("_coin", "_elo"))
print(f"Coin-flip vs Elo-weighted remainder of the season: correlation = {cmp3.stake_coin.corr(cmp3.stake_elo):.3f}; mean stake {cmp3.stake_coin.mean():.3f} vs {cmp3.stake_elo.mean():.3f}.")
print("Elo spread across teams (typical gap between the strongest and weakest side in a season):",
      round(elo.groupby(elo.match_id.map(matches.set_index('match_id').year)).elo.agg(lambda s: s.max() - s.min()).mean(), 0), "rating points")
print("\nReading: high agreement means the ranking of 'which matches matter' does not depend on the simplifying assumptions.")

Two simulation runs: correlation of match stakes = 0.999; average absolute difference = 0.0015 title points (typical stake 0.060).
Coin-flip vs Elo-weighted remainder of the season: correlation = 0.979; mean stake 0.060 vs 0.056.
Elo spread across teams (typical gap between the strongest and weakest side in a season): 258.0 rating points

Reading: high agreement means the ranking of 'which matches matter' does not depend on the simplifying assumptions.


### Cell 5 · Robustness checks

**Procedure**
1. Re-ran the whole simulation with a different random seed and compared the stakes.
2. Re-ran it with team strength (Elo ratings from notebook 02) deciding the remaining matches instead of coin flips.

**Observations**
- Two simulation runs agree at a correlation of 0.999, and the average difference is 0.0015 title points against a typical stake of 0.060. The stakes are real numbers, not simulation noise.
- Coin flips against Elo-weighted results correlate at 0.979, and the mean stake changes only from 0.060 to 0.056. The "every match is 50-50" simplification does not change which matches matter, even though teams differ by about 258 Elo points.
- Limitation for the report: net run rate is approximated by random tie-breaks, and the chance of a no-result (rain) is ignored in future matches.

In [6]:
# ─────────────────────────────────────────────────────────────────────
# CELL 6 · PUT THE STAKES ON EVERY BALL, AND SAVE
# ─────────────────────────────────────────────────────────────────────
# FUNCTIONAL: a batter plays for the batting team, so HIS pressure includes how much that team needs this match; a bowler's includes
# the bowling team's stake. Each ball therefore gets the stake of the batting team (stake_bat) and of the bowling team (stake_bowl).
# Not-measured matches (non-knockout T20I and A-team games) stay empty here; the pressure model will treat them as neutral.
ball = ball.drop(columns=["stake_bat", "stake_bowl", "match_stake", "match_kind", "ladder", "stakes_tier"], errors="ignore")      # safe to re-run
st = rows.set_index(["match_id", "team"]).stake
for col, team_col in [("stake_bat", "batting_team_id"), ("stake_bowl", "bowling_team_id")]:
    ball[col] = pd.MultiIndex.from_arrays([ball.match_id, ball[team_col]]).map(st.to_dict().get)
ball["match_stake"] = ball[["stake_bat", "stake_bowl"]].max(axis=1)
kind = rows.drop_duplicates("match_id").set_index("match_id").kind
ball["match_kind"] = ball.match_id.map(kind)
ball["ladder"] = ball.match_id.map(rows.drop_duplicates("match_id").set_index("match_id").ladder)

# A readable tier for reports and the website (the model itself will use the numbers).
bands = [-0.01, 0.05, 0.10, 0.249, 10]
ball["stakes_tier"] = pd.cut(ball.match_stake, bands, labels=["low (<0.05)", "medium (0.05-0.10)", "high (0.10-0.25)", "sudden-death / title (0.25+)"]).astype(object)
ball["stakes_tier"] = ball.stakes_tier.fillna("not measured")
print("Balls by stakes tier:")
display(pd.concat([ball.stakes_tier.value_counts().rename("balls"), (ball.stakes_tier.value_counts(normalize=True)).rename("share"),
                   ball.groupby("stakes_tier").match_id.nunique().rename("matches")], axis=1).round(3))

# Saved table: one row per match x team, for the report, the website and notebook 05c.
match_stakes = rows.drop(columns=["win_team_id"]).rename(columns={"team": "team_id"})
match_stakes = match_stakes.merge(pd.DataFrame({"team_id": name.index, "team": name.values}), on="team_id", how="left")
print("\nSaved match_stakes:", match_stakes.shape, save_table(match_stakes, "match_stakes"))
print("Saved ball_master:", ball.shape, save_table(ball, "ball_master"))

Balls by stakes tier:


,balls,share,matches
not measured,110632,0.551,489
medium (0.05-0.10),55212,0.275,231
high (0.10-0.25),13865,0.069,58
low (<0.05),13685,0.068,57
sudden-death / title (0.25+),7567,0.038,32



Saved match_stakes: (1766, 17) parquet
Saved ball_master: (200961, 114) parquet


### Cell 6 · Stakes on every ball, and save

**Procedure**
1. Gave every ball the stake of the batting team (stake_bat) and of the bowling team (stake_bowl), plus the larger of the two (match_stake), the kind of match and the playoff ladder step.
2. Added a readable tier for reports and the website: low (<0.05), medium, high (0.10 to 0.25) and sudden-death / title (0.25+).
3. Saved match_stakes (one row per match and team) and the updated ball_master.

**Observations**
- 45% of balls are in measured matches: 3.8% in sudden-death or title matches (32 matches), 6.9% in high-stake matches (58), 27.5% medium (231) and 6.8% low (57). The remaining 55% (489 matches) are not measured and will be treated as neutral.
- A batter's pressure will use his own team's stake, and a bowler's the bowling team's.
- Plan for the pressure model: use stakes as a modest multiplier and as a separate "performance when the match mattered" component. The weight gets tested on the 2022-23 to 2024-26 split, not assumed, because the average stake is flat across the season.
- Saved: match_stakes (1,766 x 17) and ball_master (200,961 x 114).